# Gold — RF-1 Risk matrix (Susceptibility × Consequence)

## Dependencies

Library dependencies are supplied by the **geohazard_env** Fabric Environment attached to this notebook, not by inline `%pip install`. Inline installation is disabled in many tenants and fails with MagicUsageError when a notebook runs from a pipeline. See `fabric/environment/requirements.txt`.

In [ ]:
# Pipeline parameter. Blank values get a unique manual-run identifier.
PIPELINE_RUN_ID = ""

In [ ]:
# Run identity is resolved before anything is written, so every gold table, artefact,
# and handoff document for this execution carries the same immutable run_id.
import re
import uuid
from datetime import datetime, timezone

_raw_run_id = str(PIPELINE_RUN_ID or "").strip()
if not _raw_run_id:
    _raw_run_id = f"manual-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"
RUN_ID = re.sub(r"[^A-Za-z0-9._-]+", "-", _raw_run_id)[:128].strip(".-_")
if not RUN_ID:
    raise ValueError("PIPELINE_RUN_ID did not contain any file-system-safe characters.")
print(f"run_id: {RUN_ID}")

## 1. Read silver and score risk

`risk_score = S \u00d7 C` (1\u201325). Bands per the RF-1 5\u00d75 matrix:
Low 1\u20134 \u00b7 Moderate 5\u20139 \u00b7 High 10\u201319 \u00b7 Extreme 20\u201325.

In [ ]:
from pyspark.sql import functions as F

# Read through the Fabric catalog first, then use a portable all-GUID path.
SILVER_LH_NAME = "silver_lakehouse"
SILVER_SCHEMA = "dbo"
WS = notebookutils.runtime.context["currentWorkspaceId"]
SILVER_LH_ID = notebookutils.lakehouse.get(SILVER_LH_NAME, workspaceId=WS).id
ONELAKE_ENDPOINT = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
SILVER_ABFSS = f"abfss://{WS}@{ONELAKE_ENDPOINT}/{SILVER_LH_ID}/Tables"

def _quoted_identifier(*parts):
    return ".".join(f"`{str(part).replace('`', '``')}`" for part in parts if part)

def read_silver_table(table_name):
    attempts = []
    table_refs = [
        _quoted_identifier(SILVER_LH_NAME, SILVER_SCHEMA, table_name),
        _quoted_identifier(SILVER_LH_NAME, table_name),
        _quoted_identifier(SILVER_SCHEMA, table_name),
        _quoted_identifier(table_name),
    ]

    for table_ref in table_refs:
        try:
            return spark.read.table(table_ref)
        except Exception as exc:
            attempts.append(f"{table_ref}: {str(exc).splitlines()[0][:140]}")

    path = f"{SILVER_ABFSS}/{table_name}"
    try:
        return spark.read.format("delta").load(path)
    except Exception as exc:
        attempts.append(f"{path}: {str(exc).splitlines()[0][:140]}")

    raise RuntimeError("Could not read silver table " + table_name + " via:\n  " + "\n  ".join(attempts))

_silver_all = read_silver_table("silver_rf1_soil_susceptibility")

# Bind gold to one silver run. If this notebook is run standalone (no pipeline run id),
# fall back to the most recent silver run and adopt ITS run_id, so gold, the map
# artefacts, and the handoff contract all describe the same screening run.
if "run_id" in _silver_all.columns:
    if _silver_all.filter(F.col("run_id") == RUN_ID).limit(1).count() == 0:
        _latest = (_silver_all.groupBy("run_id")
                   .agg(F.max("ingested_at_utc").alias("_t"))
                   .orderBy(F.desc("_t")).first())
        if _latest is None:
            raise RuntimeError("silver_rf1_soil_susceptibility is empty; run the silver notebook first.")
        print(f"  no silver rows for run_id={RUN_ID}; adopting latest silver run {_latest['run_id']}")
        RUN_ID = _latest["run_id"]
    silver = _silver_all.filter(F.col("run_id") == RUN_ID)
else:
    print("  silver table predates run scoping; reading all rows")
    silver = _silver_all

if silver.limit(1).count() == 0:
    raise RuntimeError(f"No silver pixels found for run_id={RUN_ID}.")

band_col = (
    F.when(F.col("risk_score") <= 4, "Low")
     .when(F.col("risk_score") <= 9, "Moderate")
     .when(F.col("risk_score") <= 19, "High")
     .otherwise("Extreme")
)

risk = (silver
    .withColumn("risk_score", F.col("s_rating") * F.col("c_rating"))
    .withColumn("risk_band", band_col)
    .withColumn("run_id", F.lit(RUN_ID)))

print(f"Silver pixels read: {risk.count():,}")
risk.groupBy("risk_band").count().orderBy("risk_band").show()

## 2. Write the gold tables

- `gold_rf1_risk_pixels` \u2014 per-pixel risk score + band (with coordinates).
- `gold_rf1_risk_matrix` \u2014 the 5\u00d75 S\u00d7C grid (pixel count, mean risk, band).
- `gold_rf1_band_summary` \u2014 area (km\u00b2) and share per risk band.

In [ ]:
def write_run_scoped(dataframe, table_name):
    """Dynamic partition overwrite: replace only this run, keep earlier runs.

    Every gold table carries run_id so the published data agent can honour its
    "filter every query by the run identifier" instruction.
    """
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
            .saveAsTable(table_name))
    except Exception as error:
        # A schema change cannot be applied in dynamic partition overwrite mode:
        # Delta rejects overwriteSchema with DELTA_OVERWRITE_SCHEMA_WITH_DYNAMIC_
        # PARTITION_OVERWRITE. Drop back to a static full overwrite, which replaces
        # every run, then restore dynamic mode for subsequent writes.
        print(f"  WARNING: {table_name} schema changed - replacing ALL runs. "
              f"({str(error).splitlines()[0][:120]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


# --- 1) per-pixel risk ---
# Carries the source identity burned in silver (soil_poly_id / geology_poly_id) so the
# risk surface can be traced back to the mapped units it sits on.
_pixel_columns = [
    "run_id", "row", "col", "utm_x", "utm_y", "lon", "lat",
    "elevation_m", "slope_deg", "worldcover_class",
    "soil_soft", "soil_mapped", "soil_poly_id", "geology_poly_id", "p_soft",
    "s_rating", "c_rating", "risk_score", "risk_band",
    "aoi_name", "aoi_lat", "aoi_lon", "resolution_m", "risk_factor", "ingested_at_utc",
]
_available = [column for column in _pixel_columns if column in risk.columns]
_missing = [column for column in _pixel_columns if column not in risk.columns]
if _missing:
    print(f"  note: silver did not supply {_missing} (older silver run)")
pixels = risk.select(*_available)
write_run_scoped(pixels, "gold_rf1_risk_pixels")

# pixel ground area (silver grid is RES_M metres square) -> km2 per pixel
RES_M       = float(silver.select("resolution_m").first()["resolution_m"])
PX_AREA_KM2 = (RES_M * RES_M) / 1_000_000.0
TOTAL_PX    = risk.count()

band_expr = (
    F.when(F.col("risk_score") <= 4, "Low")
     .when(F.col("risk_score") <= 9, "Moderate")
     .when(F.col("risk_score") <= 19, "High")
     .otherwise("Extreme")
)

# --- 2) 5x5 risk matrix: pixel count + mean risk per SxC cell ---
matrix = (risk.groupBy("run_id", "s_rating", "c_rating")
    .agg(F.count(F.lit(1)).alias("pixel_count"),
         F.avg("risk_score").alias("mean_risk"))
    .withColumn("risk_score", F.col("s_rating") * F.col("c_rating"))
    .withColumn("risk_band", band_expr)
    .orderBy("c_rating", "s_rating"))
write_run_scoped(matrix, "gold_rf1_risk_matrix")

# --- 3) band summary: area (km2) and share per risk band ---
band_summary = (risk.groupBy("run_id", "risk_band")
    .agg(F.count(F.lit(1)).alias("pixel_count"))
    .withColumn("area_km2", F.col("pixel_count") * F.lit(PX_AREA_KM2))
    .withColumn("pct", F.col("pixel_count") / F.lit(TOTAL_PX) * 100.0)
    .orderBy("risk_band"))
write_run_scoped(band_summary, "gold_rf1_band_summary")

print(f"run_id               : {RUN_ID}")
print(f"gold_rf1_risk_pixels : {pixels.count():,} rows")
print(f"gold_rf1_risk_matrix : {matrix.count()} SxC cells")
print(f"gold_rf1_band_summary: {band_summary.count()} bands  (px area = {PX_AREA_KM2} km2)")

## 3. Rank the risk hotspots

Dissolved band polygons answer *how much* of the AOI is High or Extreme, but not
*where* or *why*. This step clusters contiguous High/Extreme pixels into discrete
hotspots and attaches the evidence needed to explain each one: the mapped soil unit and
its drainage class, the surficial-geology unit, dominant land cover, mean slope, and the
exact distance to the nearest mapped fault.

Ordering is deterministic (mean risk, then size, then latitude, then longitude) and the
list is capped, so the same run always produces the same ranked hotspots with the same
`hs-NNN` identifiers - which is what lets a report cite a hotspot and a map resolve it.

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

import folium
import numpy as np
import shapefile
from pyproj import CRS, Transformer
from rasterio.features import shapes as raster_shapes
from rasterio.transform import Affine
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon
from shapely.geometry import mapping as geometry_mapping
from shapely.geometry import shape as geometry_shape
from shapely.geometry.polygon import orient
from shapely.ops import transform as transform_geometry
from shapely.ops import unary_union

RISK_BAND_CODES = {"Low": 1, "Moderate": 2, "High": 3, "Extreme": 4}
CODE_TO_RISK_BAND = {value: key for key, value in RISK_BAND_CODES.items()}
RISK_BAND_COLORS = {
    "Low": "#2c7bb6",
    "Moderate": "#fdae61",
    "High": "#f46d43",
    "Extreme": "#a50026",
}
RISK_BAND_RANGES = {
    "Low": (1, 4),
    "Moderate": (5, 9),
    "High": (10, 19),
    "Extreme": (20, 25),
}

# ESA WorldCover class codes -> readable labels, so the report can name land cover
# instead of quoting an integer.
WORLDCOVER_LABELS = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}

# One collect for the whole notebook: the polygonizer, the hotspot clustering, and the
# web map all read these arrays instead of re-materialising the pixel table.
_grid_columns = [
    "row", "col", "utm_x", "utm_y", "aoi_lat", "aoi_lon", "resolution_m",
    "risk_band", "risk_score", "s_rating", "c_rating", "aoi_name", "ingested_at_utc",
    "elevation_m", "slope_deg", "worldcover_class", "soil_soft",
    "soil_poly_id", "geology_poly_id",
]
_grid_available = [column for column in _grid_columns if column in risk.columns]
risk_pdf = risk.select(*_grid_available).toPandas()
if risk_pdf.empty:
    raise RuntimeError("gold_rf1_risk_pixels is empty; no map artifacts can be generated.")
unknown_bands = sorted(set(risk_pdf["risk_band"].dropna()) - set(RISK_BAND_CODES))
if unknown_bands:
    raise ValueError(f"Unexpected risk bands: {unknown_bands}")

height = int(risk_pdf["row"].max()) + 1
width = int(risk_pdf["col"].max()) + 1
resolution_m = float(risk_pdf["resolution_m"].iloc[0])
aoi_lat = float(risk_pdf["aoi_lat"].iloc[0])
aoi_lon = float(risk_pdf["aoi_lon"].iloc[0])
aoi_name = str(risk_pdf["aoi_name"].iloc[0])
source_generated_at = str(risk_pdf["ingested_at_utc"].iloc[0])

row_index = risk_pdf["row"].to_numpy(dtype=np.int32)
col_index = risk_pdf["col"].to_numpy(dtype=np.int32)

grid = np.zeros((height, width), dtype=np.uint8)
grid[row_index, col_index] = risk_pdf["risk_band"].map(RISK_BAND_CODES).to_numpy(dtype=np.uint8)
valid_mask = grid > 0


def to_grid(column, dtype="float32", fill=np.nan):
    """Scatter a pixel column back onto the (height, width) analysis grid."""
    array = np.full((height, width), fill, dtype=dtype)
    if column in risk_pdf.columns:
        array[row_index, col_index] = risk_pdf[column].to_numpy(dtype=dtype)
    return array


origin_x = float(np.median(
    risk_pdf["utm_x"].to_numpy(dtype=float)
    - (risk_pdf["col"].to_numpy(dtype=float) + 0.5) * resolution_m
))
origin_y = float(np.median(
    risk_pdf["utm_y"].to_numpy(dtype=float)
    + (risk_pdf["row"].to_numpy(dtype=float) + 0.5) * resolution_m
))
if not np.isfinite(origin_x) or not np.isfinite(origin_y):
    raise ValueError("Could not reconstruct a finite raster origin from silver pixel coordinates.")
grid_transform = Affine(resolution_m, 0.0, origin_x, 0.0, -resolution_m, origin_y)

utm_zone = min(60, max(1, int((aoi_lon + 180.0) // 6.0) + 1))
utm_epsg = (32600 if aoi_lat >= 0 else 32700) + utm_zone
utm_crs = CRS.from_epsg(utm_epsg)
wgs84_crs = CRS.from_epsg(4326)
to_wgs84 = Transformer.from_crs(utm_crs, wgs84_crs, always_xy=True)
to_utm = Transformer.from_crs(wgs84_crs, utm_crs, always_xy=True)


def iter_polygons(geometry):
    if isinstance(geometry, Polygon):
        yield geometry
    elif isinstance(geometry, MultiPolygon):
        yield from geometry.geoms
    elif isinstance(geometry, GeometryCollection):
        for part in geometry.geoms:
            yield from iter_polygons(part)


print(f"Materialised {len(risk_pdf):,} pixels onto a {height} x {width} grid "
      f"@ {resolution_m:g} m ({utm_crs.to_string()})")

In [ ]:
from scipy import ndimage

TOP_N_HOTSPOTS = 25
MIN_HOTSPOT_PIXELS = 25          # 25 px @ 10 m = 2500 m2; below this is speckle
HOTSPOT_BANDS = ("High", "Extreme")

score_grid_px = to_grid("risk_score")
s_grid = to_grid("s_rating")
c_grid = to_grid("c_rating")
slope_grid = to_grid("slope_deg")
elevation_grid = to_grid("elevation_m")
worldcover_grid = to_grid("worldcover_class")
soil_soft_grid = to_grid("soil_soft")
soil_id_grid = to_grid("soil_poly_id", fill=0.0)
geology_id_grid = to_grid("geology_poly_id", fill=0.0)


def _mode(values):
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return None
    counts = np.bincount(finite.astype(np.int64))
    return int(np.argmax(counts))


# --- attribute lookups: poly_id -> bronze feature -> typed attributes -------------
def _load_lookups():
    poly_to_key, key_to_attributes = {}, {}
    try:
        lookup = read_silver_table("silver_rf1_poly_lookup")
        if "run_id" in lookup.columns:
            lookup = lookup.filter(F.col("run_id") == RUN_ID)
        for row in lookup.select("layer", "poly_id", "feature_key").collect():
            poly_to_key[(row["layer"], int(row["poly_id"]))] = row["feature_key"]
    except Exception as error:
        print(f"  (no silver_rf1_poly_lookup: {str(error).splitlines()[0][:90]})")
    try:
        features = read_silver_table("silver_source_features")
        if "run_id" in features.columns:
            latest_run = (features.groupBy("run_id")
                          .agg(F.max("ingested_at_utc").alias("_t"))
                          .orderBy(F.desc("_t")).first())
            if latest_run is not None:
                features = features.filter(F.col("run_id") == latest_run["run_id"])
        for row in features.select("feature_key", "name", "unit_code", "drainage_class",
                                   "parent_material", "texture").collect():
            key_to_attributes[row["feature_key"]] = {
                "name": row["name"],
                "unit_code": row["unit_code"],
                "drainage_class": row["drainage_class"],
                "parent_material": row["parent_material"],
                "texture": row["texture"],
            }
    except Exception as error:
        print(f"  (no silver_source_features: {str(error).splitlines()[0][:90]})")
    return poly_to_key, key_to_attributes


POLY_TO_KEY, KEY_TO_ATTRIBUTES = _load_lookups()


def _dominant_feature(identity_values, layer):
    poly_id = _mode(identity_values[identity_values > 0]) if (identity_values > 0).any() else None
    if not poly_id:
        return None, None, {}
    feature_key = POLY_TO_KEY.get((layer, int(poly_id)))
    return feature_key, poly_id, KEY_TO_ATTRIBUTES.get(feature_key or "", {})


# --- exact nearest mapped fault, computed from bronze fault geometry -------------
def _load_fault_geometries():
    geometries = []
    try:
        from shapely import wkt as shapely_wkt
        features = read_silver_table("silver_source_features").filter(
            F.col("feature_class") == "fault")
        for row in features.select("geometry_wkt").collect():
            if not row["geometry_wkt"]:
                continue
            try:
                geometries.append(transform_geometry(
                    to_utm.transform, shapely_wkt.loads(row["geometry_wkt"])))
            except Exception:
                continue
    except Exception as error:
        print(f"  (no fault geometry available: {str(error).splitlines()[0][:90]})")
    return geometries


FAULT_GEOMETRIES = _load_fault_geometries()
print(f"  fault geometries available for distance: {len(FAULT_GEOMETRIES)}")

# --- cluster contiguous High/Extreme pixels --------------------------------------
# A "hotspot" covering most of the AOI is not a hotspot. Where the elevated bands
# dominate the grid, tighten the threshold until the candidate area is a usable
# fraction, then fall back to a score percentile if the bands alone cannot separate.
MAX_HOTSPOT_COVERAGE = 0.25

hotspot_mask = np.isin(grid, [RISK_BAND_CODES[band] for band in HOTSPOT_BANDS])
grid_cells = int(valid_mask.sum()) or 1
coverage = hotspot_mask.sum() / grid_cells
if coverage > MAX_HOTSPOT_COVERAGE:
    tightened = np.isin(grid, [RISK_BAND_CODES["Extreme"]])
    tightened_coverage = tightened.sum() / grid_cells
    print(f"  High+Extreme covers {coverage:.1%} of the grid; tightening to Extreme "
          f"({tightened_coverage:.1%})")
    hotspot_mask = tightened
    coverage = tightened_coverage
if coverage > MAX_HOTSPOT_COVERAGE:
    finite_scores = score_grid_px[np.isfinite(score_grid_px)]
    cutoff = float(np.quantile(finite_scores, 1.0 - MAX_HOTSPOT_COVERAGE))
    hotspot_mask = np.isfinite(score_grid_px) & (score_grid_px >= cutoff)
    print(f"  still {coverage:.1%}; using the top {MAX_HOTSPOT_COVERAGE:.0%} of risk "
          f"scores (score >= {cutoff:.0f})")

labels, cluster_count = ndimage.label(hotspot_mask, structure=np.ones((3, 3), dtype=int))
print(f"  {cluster_count} contiguous {'/'.join(HOTSPOT_BANDS)} clusters found")

candidates = []
for cluster_id in range(1, cluster_count + 1):
    cluster = labels == cluster_id
    pixel_count = int(cluster.sum())
    if pixel_count < MIN_HOTSPOT_PIXELS:
        continue

    cluster_rows, cluster_cols = np.nonzero(cluster)
    utm_x_values = origin_x + (cluster_cols + 0.5) * resolution_m
    utm_y_values = origin_y - (cluster_rows + 0.5) * resolution_m
    centroid_x = float(utm_x_values.mean())
    centroid_y = float(utm_y_values.mean())
    centroid_lon, centroid_lat = to_wgs84.transform(centroid_x, centroid_y)

    corner_lons, corner_lats = to_wgs84.transform(
        np.array([utm_x_values.min(), utm_x_values.max()]),
        np.array([utm_y_values.min(), utm_y_values.max()]),
    )

    geometry_utm = unary_union([
        geometry_shape(shape_geometry)
        for shape_geometry, _ in raster_shapes(
            cluster.astype(np.uint8), mask=cluster,
            transform=grid_transform, connectivity=8)
    ])
    if not geometry_utm.is_valid:
        geometry_utm = geometry_utm.buffer(0)
    geometry_wgs84 = transform_geometry(
        to_wgs84.transform, geometry_utm.simplify(resolution_m, preserve_topology=True))

    s_mode = _mode(s_grid[cluster])
    c_mode = _mode(c_grid[cluster])
    risk_score = int(s_mode * c_mode) if (s_mode and c_mode) else int(
        np.nanmax(score_grid_px[cluster]))
    band = next(band for band, (low, high) in RISK_BAND_RANGES.items()
                if low <= risk_score <= high)

    soil_key, soil_poly_id, soil_attributes = _dominant_feature(soil_id_grid[cluster], "soil_survey")
    geology_key, geology_poly_id, geology_attributes = _dominant_feature(
        geology_id_grid[cluster], "surficial_geology")

    nearest_fault_km = None
    if FAULT_GEOMETRIES:
        from shapely.geometry import Point as ShapelyPoint
        centre = ShapelyPoint(centroid_x, centroid_y)
        nearest_fault_km = float(min(geometry.distance(centre)
                                     for geometry in FAULT_GEOMETRIES)) / 1000.0

    worldcover_mode = _mode(worldcover_grid[cluster])
    candidates.append({
        "run_id": RUN_ID,
        "pixel_count": pixel_count,
        "area_km2": float(pixel_count) * (resolution_m ** 2) / 1_000_000.0,
        "mean_risk_score": float(np.nanmean(score_grid_px[cluster])),
        "max_risk_score": int(np.nanmax(score_grid_px[cluster])),
        "s_rating": int(s_mode) if s_mode else None,
        "c_rating": int(c_mode) if c_mode else None,
        "risk_score": risk_score,
        "risk_band": band,
        "centroid_lon": float(centroid_lon),
        "centroid_lat": float(centroid_lat),
        "bbox_minx": float(min(corner_lons)),
        "bbox_miny": float(min(corner_lats)),
        "bbox_maxx": float(max(corner_lons)),
        "bbox_maxy": float(max(corner_lats)),
        "mean_slope_deg": float(np.nanmean(slope_grid[cluster])),
        "mean_elevation_m": float(np.nanmean(elevation_grid[cluster])),
        "mean_soil_soft": float(np.nanmean(soil_soft_grid[cluster])),
        "worldcover_class": worldcover_mode,
        "worldcover_label": WORLDCOVER_LABELS.get(worldcover_mode) if worldcover_mode else None,
        "soil_feature_key": soil_key,
        "soil_name": soil_attributes.get("name"),
        "soil_drainage_class": soil_attributes.get("drainage_class"),
        "soil_parent_material": soil_attributes.get("parent_material"),
        "geology_feature_key": geology_key,
        "geology_name": geology_attributes.get("name"),
        "geology_unit_code": geology_attributes.get("unit_code"),
        "nearest_fault_km": nearest_fault_km,
        "aoi_name": aoi_name,
        "generated_at_utc": source_generated_at,
        "_geometry": geometry_wgs84,
    })

# Deterministic ordering: mean risk, then size, then position. Ties can never reorder.
candidates.sort(key=lambda item: (
    -item["mean_risk_score"], -item["pixel_count"],
    item["centroid_lat"], item["centroid_lon"],
))
if len(candidates) > TOP_N_HOTSPOTS:
    print(f"  capping {len(candidates)} qualifying clusters at the top {TOP_N_HOTSPOTS}")
hotspots = candidates[:TOP_N_HOTSPOTS]

hotspot_features = []
hotspot_rows = []
for rank, hotspot in enumerate(hotspots, start=1):
    feature_id = f"hs-{rank:03d}"
    geometry = hotspot.pop("_geometry")
    record = {"hotspot_id": feature_id, "rank": rank, **hotspot}
    record["geometry_json"] = json.dumps(geometry_mapping(geometry), separators=(",", ":"))
    hotspot_rows.append(record)
    hotspot_features.append({
        "type": "Feature",
        "id": feature_id,
        "properties": {key: value for key, value in record.items() if key != "geometry_json"},
        "geometry": geometry_mapping(geometry),
    })

# An explicit schema is required, not optional: when no soil or geology polygon
# underlies any hotspot, those columns are entirely None and Spark's type inference
# fails with CANNOT_DETERMINE_TYPE rather than defaulting to string.
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType,
)

HOTSPOT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("hotspot_id", StringType(), True),
    StructField("rank", IntegerType(), True),
    StructField("pixel_count", IntegerType(), True),
    StructField("area_km2", DoubleType(), True),
    StructField("mean_risk_score", DoubleType(), True),
    StructField("max_risk_score", IntegerType(), True),
    StructField("s_rating", IntegerType(), True),
    StructField("c_rating", IntegerType(), True),
    StructField("risk_score", IntegerType(), True),
    StructField("risk_band", StringType(), True),
    StructField("centroid_lon", DoubleType(), True),
    StructField("centroid_lat", DoubleType(), True),
    StructField("bbox_minx", DoubleType(), True),
    StructField("bbox_miny", DoubleType(), True),
    StructField("bbox_maxx", DoubleType(), True),
    StructField("bbox_maxy", DoubleType(), True),
    StructField("mean_slope_deg", DoubleType(), True),
    StructField("mean_elevation_m", DoubleType(), True),
    StructField("mean_soil_soft", DoubleType(), True),
    StructField("worldcover_class", IntegerType(), True),
    StructField("worldcover_label", StringType(), True),
    StructField("soil_feature_key", StringType(), True),
    StructField("soil_name", StringType(), True),
    StructField("soil_drainage_class", StringType(), True),
    StructField("soil_parent_material", StringType(), True),
    StructField("geology_feature_key", StringType(), True),
    StructField("geology_name", StringType(), True),
    StructField("geology_unit_code", StringType(), True),
    StructField("nearest_fault_km", DoubleType(), True),
    StructField("aoi_name", StringType(), True),
    StructField("generated_at_utc", StringType(), True),
    StructField("geometry_json", StringType(), True),
])

if hotspot_rows:
    _hotspot_columns = [field.name for field in HOTSPOT_SCHEMA.fields]
    _hotspot_records = [
        tuple(row.get(column) for column in _hotspot_columns) for row in hotspot_rows
    ]
    hotspots_sdf = spark.createDataFrame(_hotspot_records, schema=HOTSPOT_SCHEMA)
    write_run_scoped(hotspots_sdf, "gold_rf1_risk_hotspots")
    print(f"\ngold_rf1_risk_hotspots: {len(hotspot_rows)} ranked hotspots")
    (spark.read.table("gold_rf1_risk_hotspots").filter(F.col("run_id") == RUN_ID)
        .select("rank", "risk_band", "risk_score", "area_km2", "centroid_lat", "centroid_lon",
                "soil_drainage_class", "worldcover_label", "nearest_fault_km")
        .orderBy("rank").show(10, truncate=False))
else:
    print("\nNo qualifying High/Extreme clusters; gold_rf1_risk_hotspots not written.")

## 4. Publish risk-area polygons, hotspots, and the web map

Dissolve contiguous risk bands from the 10 m grid into bounded MultiPolygon features and
publish the run-scoped artifact set: a Delta table, band GeoJSON, **hotspot GeoJSON**,
Shapefile bundle, manifest, and an interactive HTML map. `_SUCCESS` is written last, only
after every artifact is non-empty and the persisted GeoJSON and manifest pass structural
checks, so a consumer can never read a half-published run.

In [ ]:
OUTPUT_ROOT = Path("/lakehouse/default/Files/gold_rf1_webmap/runs")
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEOJSON_PATH = OUTPUT_DIR / "gold_rf1_risk_areas.geojson"
HOTSPOT_GEOJSON_PATH = OUTPUT_DIR / "gold_rf1_risk_hotspots.geojson"
SHAPEFILE_BASE = OUTPUT_DIR / "gold_rf1_risk_areas"
SHAPEFILE_ZIP_PATH = OUTPUT_DIR / "gold_rf1_risk_areas_shapefile.zip"
WEBMAP_PATH = OUTPUT_DIR / "gold_rf1_webmap.html"
MANIFEST_PATH = OUTPUT_DIR / "gold_rf1_webmap_manifest.json"
SUCCESS_PATH = OUTPUT_DIR / "_SUCCESS"

# --- dissolve the 10 m grid into one MultiPolygon per populated risk band ----------
polygons_by_band = {band: [] for band in RISK_BAND_CODES}
for raw_geometry, raw_code in raster_shapes(
    grid, mask=valid_mask, transform=grid_transform, connectivity=8,
):
    polygons_by_band[CODE_TO_RISK_BAND[int(raw_code)]].append(geometry_shape(raw_geometry))

features = []
area_rows = []
for band in RISK_BAND_CODES:
    source_polygons = polygons_by_band[band]
    if not source_polygons:
        continue

    merged_utm = unary_union(source_polygons)
    if not merged_utm.is_valid:
        merged_utm = merged_utm.buffer(0)
    simplified_utm = merged_utm.simplify(resolution_m, preserve_topology=True)
    geometry_wgs84 = transform_geometry(to_wgs84.transform, simplified_utm)

    pixel_count = int((risk_pdf["risk_band"] == band).sum())
    area_km2 = float(merged_utm.area / 1_000_000.0)
    percent = float(pixel_count / len(risk_pdf) * 100.0)
    risk_min, risk_max = RISK_BAND_RANGES[band]
    feature_id = f"rf1-{band.lower()}"
    geometry_json = geometry_mapping(geometry_wgs84)
    properties = {
        "feature_id": feature_id,
        "run_id": RUN_ID,
        "risk_band": band,
        "risk_min": risk_min,
        "risk_max": risk_max,
        "pixel_count": pixel_count,
        "area_km2": round(area_km2, 6),
        "pct": round(percent, 6),
        "aoi_name": aoi_name,
        "source_table": "gold_rf1_risk_pixels",
        "generated_at_utc": source_generated_at,
    }
    features.append({"type": "Feature", "id": feature_id, "properties": properties, "geometry": geometry_json})
    area_rows.append({**properties, "geometry_json": json.dumps(geometry_json, separators=(",", ":"))})

if not features:
    raise RuntimeError("Risk-band polygonization produced no features.")
if len(features) > len(RISK_BAND_CODES):
    raise RuntimeError("Risk-area GeoJSON exceeded the one-feature-per-band limit.")

feature_collection = {
    "type": "FeatureCollection",
    "name": "gold_rf1_risk_areas",
    "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
    "features": features,
}
with GEOJSON_PATH.open("w", encoding="utf-8") as stream:
    json.dump(feature_collection, stream, ensure_ascii=True, separators=(",", ":"))

write_run_scoped(spark.createDataFrame(area_rows), "gold_rf1_risk_areas")

# --- ranked hotspots as a separate, bounded GeoJSON layer -------------------------
hotspot_collection = {
    "type": "FeatureCollection",
    "name": "gold_rf1_risk_hotspots",
    "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
    "features": hotspot_features,
}
with HOTSPOT_GEOJSON_PATH.open("w", encoding="utf-8") as stream:
    json.dump(hotspot_collection, stream, ensure_ascii=True, separators=(",", ":"), default=str)

# --- Shapefile bundle for GIS hand-off --------------------------------------------
writer = shapefile.Writer(str(SHAPEFILE_BASE), shapeType=shapefile.POLYGON, encoding="utf-8")
writer.field("FEATURE_ID", "C", size=32)
writer.field("RISK_BAND", "C", size=16)
writer.field("RISK_MIN", "N", size=3, decimal=0)
writer.field("RISK_MAX", "N", size=3, decimal=0)
writer.field("PIXELS", "N", size=12, decimal=0)
writer.field("AREA_KM2", "F", size=18, decimal=6)
writer.field("PCT", "F", size=12, decimal=6)
writer.field("AOI_NAME", "C", size=80)

for feature in features:
    geometry = geometry_shape(feature["geometry"])
    parts = []
    for polygon in iter_polygons(geometry):
        oriented = orient(polygon, sign=-1.0)
        parts.append(list(oriented.exterior.coords))
        parts.extend(list(interior.coords) for interior in oriented.interiors)
    writer.poly(parts)
    props = feature["properties"]
    writer.record(
        props["feature_id"], props["risk_band"], props["risk_min"], props["risk_max"],
        props["pixel_count"], props["area_km2"], props["pct"], props["aoi_name"],
    )
writer.close()

(SHAPEFILE_BASE.with_suffix(".prj")).write_text(wgs84_crs.to_wkt(version="WKT1_ESRI"), encoding="ascii")
(SHAPEFILE_BASE.with_suffix(".cpg")).write_text("UTF-8", encoding="ascii")
with zipfile.ZipFile(SHAPEFILE_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for suffix in (".shp", ".shx", ".dbf", ".prj", ".cpg"):
        component = SHAPEFILE_BASE.with_suffix(suffix)
        archive.write(component, arcname=component.name)

# --- manifest ---------------------------------------------------------------------
generated_at_utc = datetime.now(timezone.utc).isoformat()
query_hash = hashlib.sha256(json.dumps({
    "sourceTable": "gold_rf1_risk_pixels",
    "sourceGeneratedAtUtc": source_generated_at,
    "pixelCount": int(len(risk_pdf)),
    "features": [feature["properties"]["feature_id"] for feature in features],
}, sort_keys=True).encode("utf-8")).hexdigest()
hotspot_hash = hashlib.sha256(json.dumps({
    "sourceTable": "gold_rf1_risk_hotspots",
    "runId": RUN_ID,
    "features": [feature["id"] for feature in hotspot_features],
}, sort_keys=True).encode("utf-8")).hexdigest()

all_geometry = unary_union([geometry_shape(feature["geometry"]) for feature in features])
min_lon, min_lat, max_lon, max_lat = all_geometry.bounds
relative_output_dir = f"Files/gold_rf1_webmap/runs/{RUN_ID}"
manifest = {
    "schemaVersion": "1.0",
    "runId": RUN_ID,
    "generatedAtUtc": generated_at_utc,
    "viewport": {
        "center": [aoi_lon, aoi_lat],
        "bounds": [[min_lon, min_lat], [max_lon, max_lat]],
        "zoom": 11,
    },
    "layers": [
        {
            "id": "basemap",
            "title": "OpenStreetMap",
            "role": "basemap",
            "sourceType": "rasterTiles",
            "uri": "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
            "defaultVisible": True,
            "opacity": 1.0,
            "provenance": {
                "sourceTable": "OpenStreetMap",
                "generatedAtUtc": generated_at_utc,
                "queryHash": hashlib.sha256(b"OpenStreetMap standard tiles").hexdigest(),
            },
        },
        {
            "id": "risk-areas",
            "title": "RF-1 risk areas",
            "role": "risk",
            "sourceType": "geojson",
            "uri": f"{relative_output_dir}/{GEOJSON_PATH.name}",
            "defaultVisible": True,
            "opacity": 0.72,
            "legendId": "risk-band",
            "featureIdProperty": "feature_id",
            "provenance": {
                "sourceTable": "gold_rf1_risk_pixels",
                "generatedAtUtc": generated_at_utc,
                "queryHash": query_hash,
            },
        },
        {
            "id": "hotspots",
            "title": "RF-1 ranked hotspots",
            "role": "hotspots",
            "sourceType": "geojson",
            "uri": f"{relative_output_dir}/{HOTSPOT_GEOJSON_PATH.name}",
            "defaultVisible": True,
            "opacity": 0.9,
            "legendId": "risk-band",
            "featureIdProperty": "hotspot_id",
            "featureCount": len(hotspot_features),
            "provenance": {
                "sourceTable": "gold_rf1_risk_hotspots",
                "generatedAtUtc": generated_at_utc,
                "queryHash": hotspot_hash,
            },
        },
    ],
    "legends": [{
        "id": "risk-band",
        "title": "RF-1 risk band",
        "items": [
            {"value": band, "label": band, "color": RISK_BAND_COLORS[band]}
            for band in RISK_BAND_CODES
        ],
    }],
}
with MANIFEST_PATH.open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, ensure_ascii=True, indent=2)

# --- folium map -------------------------------------------------------------------
basemap = manifest["layers"][0]
risk_map = folium.Map(location=[aoi_lat, aoi_lon], zoom_start=11, tiles=None, control_scale=True)
folium.TileLayer(
    tiles=basemap["uri"],
    attr="&copy; OpenStreetMap contributors",
    name=basemap["title"],
    overlay=False,
    control=True,
    opacity=basemap["opacity"],
).add_to(risk_map)
folium.GeoJson(
    feature_collection,
    name="RF-1 risk areas",
    style_function=lambda feature: {
        "fillColor": RISK_BAND_COLORS[feature["properties"]["risk_band"]],
        "color": "#202124",
        "weight": 1,
        "fillOpacity": 0.72,
    },
    highlight_function=lambda _feature: {"weight": 3, "fillOpacity": 0.9},
    tooltip=folium.GeoJsonTooltip(
        fields=["risk_band", "risk_min", "risk_max", "area_km2", "pct", "pixel_count"],
        aliases=["Risk band", "Minimum score", "Maximum score", "Area (km2)", "Share (%)", "Pixels"],
        localize=True,
        sticky=False,
    ),
).add_to(risk_map)

if hotspot_features:
    hotspot_layer = folium.FeatureGroup(name=f"Ranked hotspots (top {len(hotspot_features)})")
    folium.GeoJson(
        hotspot_collection,
        style_function=lambda _feature: {
            "fillColor": "#00000000", "color": "#111111", "weight": 2, "fillOpacity": 0.0,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["hotspot_id", "rank", "risk_band", "risk_score", "area_km2",
                    "soil_drainage_class", "worldcover_label"],
            aliases=["Hotspot", "Rank", "Band", "Score", "Area (km2)",
                     "Soil drainage", "Land cover"],
            localize=True,
        ),
    ).add_to(hotspot_layer)
    for record in hotspot_rows:
        detail = [
            f"<b>{record['hotspot_id']}</b> (rank {record['rank']})",
            f"{record['risk_band']} - score {record['risk_score']}",
            f"{record['area_km2']:.4f} km2 over {record['pixel_count']:,} px",
        ]
        if record.get("soil_name"):
            detail.append(f"Soil: {record['soil_name']}")
        if record.get("soil_drainage_class"):
            detail.append(f"Drainage: {record['soil_drainage_class']}")
        if record.get("geology_name"):
            detail.append(f"Surficial geology: {record['geology_name']}")
        if record.get("worldcover_label"):
            detail.append(f"Land cover: {record['worldcover_label']}")
        if record.get("nearest_fault_km") is not None:
            detail.append(f"Nearest mapped fault: {record['nearest_fault_km']:.2f} km")
        folium.Marker(
            [record["centroid_lat"], record["centroid_lon"]],
            tooltip=f"{record['hotspot_id']} - rank {record['rank']}",
            popup=folium.Popup("<br>".join(detail), max_width=320),
            icon=folium.Icon(color="black", icon="exclamation-sign"),
        ).add_to(hotspot_layer)
    hotspot_layer.add_to(risk_map)

folium.Marker(
    [aoi_lat, aoi_lon],
    tooltip=aoi_name,
    popup=f"{aoi_name}<br>RF-1 risk-area polygons",
).add_to(risk_map)
risk_map.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])
folium.LayerControl(collapsed=False).add_to(risk_map)
legend = manifest["legends"][0]
legend_items = "".join(
    f'<div><span style="display:inline-block;width:12px;height:12px;margin-right:6px;background:{item["color"]};"></span>{item["label"]}</div>'
    for item in legend["items"]
)
legend_html = (
    '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;'
    'background:#fff;border:1px solid #555;padding:10px 12px;line-height:1.5;">'
    f'<strong>{legend["title"]}</strong>{legend_items}</div>'
)
risk_map.get_root().html.add_child(folium.Element(legend_html))
risk_map.save(str(WEBMAP_PATH))

# --- validate, then write the completion marker last ------------------------------
artifacts = {
    "geojson": GEOJSON_PATH,
    "hotspotGeojson": HOTSPOT_GEOJSON_PATH,
    "shapefile": SHAPEFILE_BASE.with_suffix(".shp"),
    "shapefileZip": SHAPEFILE_ZIP_PATH,
    "manifest": MANIFEST_PATH,
    "webmap": WEBMAP_PATH,
}
for name, artifact in artifacts.items():
    if not artifact.exists() or artifact.stat().st_size <= 0:
        raise RuntimeError(f"Artifact {name} was not written correctly: {artifact}")
with GEOJSON_PATH.open("r", encoding="utf-8") as stream:
    persisted_geojson = json.load(stream)
if persisted_geojson.get("type") != "FeatureCollection" or len(persisted_geojson.get("features", [])) != len(features):
    raise RuntimeError("Persisted GeoJSON failed feature-collection validation.")
if manifest["schemaVersion"] != "1.0" or len(manifest["layers"]) < 2 or not manifest["legends"]:
    raise RuntimeError("Web-map manifest failed required-field validation.")

success = {
    "schemaVersion": "1.0",
    "runId": RUN_ID,
    "generatedAtUtc": generated_at_utc,
    "featureCount": len(features),
    "hotspotCount": len(hotspot_features),
    "artifacts": {
        name: {
            "uri": f"{relative_output_dir}/{artifact.name}",
            "sizeBytes": artifact.stat().st_size,
        }
        for name, artifact in artifacts.items()
    },
}
SUCCESS_PATH.write_text(json.dumps(success, ensure_ascii=True, indent=2), encoding="utf-8")

print(f"Wrote gold_rf1_risk_areas -> {len(features)} risk-band features")
print(f"Wrote hotspot layer      -> {len(hotspot_features)} ranked hotspots")
print(f"Run-specific output: {relative_output_dir}")
for name, artifact in {**artifacts, "completionMarker": SUCCESS_PATH}.items():
    print(f"  {name}: {artifact} ({artifact.stat().st_size:,} bytes)")

display(risk_map)

## 5. Visualise the Risk Matrix, Grid, and Band Shares

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

BAND_COLORS = {"Low": "#2c7bb6", "Moderate": "#fdae61", "High": "#f46d43", "Extreme": "#a50026"}

_run = F.col("run_id") == RUN_ID
mdf = spark.read.table("gold_rf1_risk_matrix").filter(_run).toPandas()
bdf = spark.read.table("gold_rf1_band_summary").filter(_run).toPandas()
pdf = spark.read.table("gold_rf1_risk_pixels").filter(_run).toPandas()

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))

# --- (a) 5x5 risk matrix coloured by band ---
def band_of(v):
    return "Low" if v <= 4 else "Moderate" if v <= 9 else "High" if v <= 19 else "Extreme"

cmap = ListedColormap([BAND_COLORS[b] for b in ["Low", "Moderate", "High", "Extreme"]])
norm = BoundaryNorm([1, 5, 10, 20, 26], cmap.N)
score_grid = np.zeros((5, 5), dtype=int)
count_grid = np.zeros((5, 5), dtype=int)
for _, r in mdf.iterrows():
    score_grid[int(r.c_rating) - 1, int(r.s_rating) - 1] = int(r.risk_score)
    count_grid[int(r.c_rating) - 1, int(r.s_rating) - 1] = int(r.pixel_count)

ax[0].imshow(score_grid, cmap=cmap, norm=norm, origin="lower", extent=[0.5, 5.5, 0.5, 5.5])
for s in range(1, 6):
    for c in range(1, 6):
        sc = s * c
        ax[0].text(s, c, f"{sc}\n({count_grid[c-1, s-1]:,})",
                   ha="center", va="center", fontsize=9,
                   color="white" if sc >= 10 else "black")
ax[0].set_xticks(range(1, 6)); ax[0].set_yticks(range(1, 6))
ax[0].set_xlabel("Susceptibility (S)"); ax[0].set_ylabel("Consequence (C)")
ax[0].set_title("RF-1 risk matrix  (S \u00d7 C \u2192 score, pixel count)")

# --- (b) spatial risk map ---
H = int(pdf["row"].max()) + 1
W = int(pdf["col"].max()) + 1
rmap = np.full((H, W), np.nan)
rmap[pdf["row"].values, pdf["col"].values] = pdf["risk_score"].values
im = ax[1].imshow(rmap, cmap="RdYlGn_r", vmin=1, vmax=25, origin="upper")
ax[1].set_title("Risk score over AOI (10 m)")
ax[1].set_xlabel("col"); ax[1].set_ylabel("row")
fig.colorbar(im, ax=ax[1], shrink=0.8, label="risk score")

# --- (c) band shares ---
order = ["Low", "Moderate", "High", "Extreme"]
bdf = bdf.set_index("risk_band").reindex(order).fillna(0).reset_index()
ax[2].bar(bdf["risk_band"], bdf["area_km2"],
          color=[BAND_COLORS[b] for b in bdf["risk_band"]])
for i, r in bdf.iterrows():
    ax[2].text(i, r.area_km2, f"{r.area_km2:.2f} km\u00b2\n{r.pct:.1f}%",
               ha="center", va="bottom", fontsize=9)
ax[2].set_ylabel("area (km\u00b2)"); ax[2].set_title("Area by risk band")

plt.tight_layout()
plt.show()